In [11]:
!pip uninstall -y tensorflow
!pip cache purge

Files removed: 1188 (3086.0 MB)


In [1]:
!pip install tensorflow==2.15.0

  Using cached tensorflow-2.15.0-cp39-cp39-win_amd64.whl.metadata (3.6 kB)
  Using cached tensorflow_intel-2.15.0-cp39-cp39-win_amd64.whl.metadata (5.1 kB)
Using cached tensorflow-2.15.0-cp39-cp39-win_amd64.whl (2.1 kB)
Using cached tensorflow_intel-2.15.0-cp39-cp39-win_amd64.whl (300.8 MB)



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: C:\Users\Lenovo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Load Data

In [ ]:
import numpy as np

loaded = np.load("ecg_dataset.npz")
X = loaded["x"]
y = loaded["y"]

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Model Creation

In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np


def build_cnn_model(input_shape=(5000, 12)): # (time_steps, channels)
    model = models.Sequential([
        layers.Conv1D(64, 5, activation='relu', padding='same', input_shape=input_shape),
        layers.MaxPooling1D(2),
        layers.Conv1D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling1D(2),
        layers.Conv1D(256, 3, activation='relu', padding='same'),
        layers.GlobalAveragePooling1D(),
        layers.Dense(128, activation='relu'),
    ])

    return model

cnn_model = build_cnn_model()  
cnn_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])


In [8]:
from tensorflow.keras import Model
from tensorflow.keras.layers import Dense

# Clone base
feature_extractor = cnn_model

# Add output layer (e.g., binary classification for heart failure)
output = Dense(1, activation='sigmoid')(feature_extractor.output)
training_model = Model(inputs=feature_extractor.input, outputs=output)

training_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train Model

In [9]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
        monitor='val_loss',       # Track validation loss
        patience=3,               # Stop after 3 epochs with no improvement
        restore_best_weights=True
    )
    

In [ ]:
training_model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stop])

Epoch 1/100
340/340 [==============================] - 301s 881ms/step - loss: 0.2827 - accuracy: 0.8840 - val_loss: 0.2817 - val_accuracy: 0.8801
Epoch 2/100
340/340 [==============================] - 301s 885ms/step - loss: 0.2496 - accuracy: 0.8966 - val_loss: 0.3053 - val_accuracy: 0.8801
Epoch 3/100
340/340 [==============================] - 299s 878ms/step - loss: 0.2457 - accuracy: 0.8973 - val_loss: 0.2921 - val_accuracy: 0.8842
Epoch 4/100
340/340 [==============================] - 293s 862ms/step - loss: 0.2367 - accuracy: 0.9020 - val_loss: 0.2905 - val_accuracy: 0.8941
